# Code-Beispiel 1: Zeitintegration

Die Themen dieses Code-Beispiels sind:
* Explizites Euler-Verfahren
* Implizites Euler-Verfahren
* Symplektisches Euler-Verfahren
* Trapezregel
* Konservativität der Zeitintegrationsverfahren
* Konvergenzanalyse

## Anfangswertproblem: mathematisches Pendel

![alt text](./graphics/pendel.png)

Unter Annahme eines masselosen Seils und einer konzetrierten Punktmasse kann ein Pendel mit der folgenden DGL besrieben werden:
$$
\begin{equation*}
\ddot{\varphi}(t) + \frac{g}{L}\sin(\varphi(t)) = 0
\end{equation*}
$$
Für kleine Winkel $\varphi$ kann die DGL mit $\sin(\varphi) \approx \varphi$ linearisiert werden:
$$
\begin{equation*}
\ddot{\varphi}(t) + \frac{g}{L}\varphi(t) = 0
\end{equation*}
$$
Gegeben sind außerdem die Anfangsbedingungen:
$$
\begin{aligned}
\varphi(t=0) &= \varphi_0\\
\dot{\varphi}(t=0) &= v_0.
\end{aligned}
$$
Dies ergibt insgesamt ein Anfangswertproblem.

### Analytische Lösung

Für das linearisierte Problem kann eine geschlossene Lösung gefunden werden. Wir identifizieren zunächst die Eigenkreisfrequenz des Systems:
$$
\begin{equation*}
\omega = \sqrt{\frac{g}{L}}
\end{equation*}
$$
Die Lösung des Anfangswertproblems ist gegeben durch:
$$
\begin{equation*}
\varphi(t) = \varphi_0\cos(\omega t) + \frac{v_0}{\omega} \sin(\omega t)
\end{equation*}
$$
Für das nichtlineare Problem kann keine solche Lösung angegeben werden.

### Numerische Lösung

Zur numerischen Lösung des Anfangswertproblems wird die DGL zunächst auf ein System 1. Ordnung gebracht:
$$
\begin{aligned}
\dot{\varphi}(t) &= v(t)\\
\dot{v}(t) &= -\frac{g}{L}\sin(\varphi(t))
\end{aligned}
$$

Im Folgenden werden verschiedene numerische Verfahren untersucht.

**Hinweis** : Im Code wurde $\boldsymbol{x} = \begin{pmatrix} \varphi \\ v \end{pmatrix}$ gesetzt.

### Energieerhaltung

Energie ist eine Erhaltungsgröße und damit ein Indikator für die Konservativität eines Zeitintegrationsverfahrens. Das Verfahren ist konservativ, wenn die im System enthaltene Energie konstant bleibt.

Man unterscheidet hierbei potentielle und kinetische Energie:
$$
\begin{aligned}
&E_{\mathrm{pot, lin}} &&= mgL(1-\cos(\varphi))\\
&E_{\mathrm{pot, nonl}} &&= mgL(1 - (1 - \frac{1}{2}\varphi^2)) = mgL\frac{\varphi^2}{2}\\
&E_{\mathrm{kin}} &&= \frac{1}{2}mL^2v^2\\
&E_{\mathrm{total}} &&= E_{\mathrm{pot}} + E_{\mathrm{kin}}
\end{aligned}
$$


# Implementierung

## Import

In [ ]:
# import standard libraries
import numpy as np
from scipy.optimize import fsolve
from scipy.linalg import solve
from scipy.linalg import norm
import sys

# import helper functions, depending on whether we are running in Google Colab or not

IN_COLAB = "google.colab" in sys.modules # check if we are running in Google Colab (if module exists, we are)

if IN_COLAB:
    # clone the repository to access the helper functions (the version on the main branch)
    !git clone https://github.com/CPShub/LectureNSM.git 
    from LectureNSM.ex1_time_integration.helper_functions import plot_motion, plot_energy, plot_convergence, make_animation
    %rm -rf LectureNSM  # delete the folder after importing, to enable re-importing if we run the cell again

else:
    from helper_functions import plot_motion, plot_energy, plot_convergence, make_animation

## Modellparameter

In [ ]:
g = 9.81                        # gravity acceleration in m/s^2
L = 1.0                         # pendulum length in m
delta_t = 0.05                  # time step in s
T = 10                          # simulation time in s
N = round(T/delta_t)            # number of evaluations
m = 1.0                         # mass of the pendulum in kg
omega = np.sqrt(g/L)            # angular frequency in rad/s

phi_0 = np.pi/4                 # initial angle phi in rad
v_0 = 0                         # initial velocity v in rad/s
x = np.zeros((N + 1, 2))        # preallocate array
x[0] = [phi_0, v_0]             # set initial conditions
t = np.linspace(0, T, N + 1)    # timestamps to evaluate

# analytical solution (for the linearized case)
x_ana_fun = lambda t: np.array([phi_0*np.cos(omega*t) + v_0/omega*np.sin(omega*t),
                                 -phi_0*omega*np.sin(omega*t) + v_0*np.cos(omega*t)]).T

# energy functions
E_pot_lin_fun = lambda phi: (m * g * L * (1 - (1 - 0.5*phi**2)))
E_pot_nonlin_fun = lambda phi: (m * g * L * (1 - np.cos(phi)))
E_kin_fun = lambda v: (0.5 * m * (L**2) * v**2)

# evaluate analytical solution and energies for all timestamps
x_ana = x_ana_fun(t)
E_pot_ana = E_pot_lin_fun(x_ana[:, 0])
E_kin_ana = E_kin_fun(x_ana[:, 0])

## Explizites Euler-Verfahren

Einsetzen der DGL (System erster Ordnung) in die Formeln führt auf das folgende nichtlineare Gleichungssystem 

$$
\begin{aligned}
\varphi_{n+1} &= \varphi_n + v_n \cdot \Delta t\\
v_{n+1} &= v_n - \frac{g}{L}\sin(\varphi_n) \Delta t
\end{aligned}
$$

Linearisieren von $v$ ergibt
$$
\begin{equation*}
v_{n+1} = v_n - \frac{g}{L}\varphi_n \Delta t
\end{equation*}
$$

In [ ]:

# Preallocate arrays 
x_ee_lin = x.copy()
x_ee_nonlinear = x.copy()

# Define functions f(t_k, y_k)
# Linearized Case
f_ee  = lambda x: np.array([x[1], -g/L * x[0]])

# Nonlinearized Case
f_nonlinear_ee = lambda x: np.array([x[1], -g/L * np.sin(x[0])])

# Evaluate for each time step
for n in range(N):
    # Linearized Case
    x_ee_lin[n+1] = x_ee_lin[n] + f_ee(x_ee_lin[n]) * delta_t

    # Nonlinearized Case
    x_ee_nonlinear[n+1] = x_ee_nonlinear[n] + f_nonlinear_ee(x_ee_nonlinear[n])* delta_t

# Plot results
plot_motion(t, x_ana, x_ee_lin, x_ee_nonlinear, "Explizites Euler-Verfahren")

### Energieerhaltung mit dem expliziten Euler-Verfahren

In [ ]:
# Conservation of Energy
E_pot = E_pot_lin_fun(x_ee_lin[:, 0])
E_kin = E_kin_fun(x_ee_lin[:, 1])
E_total = [e_pot + e_kin for e_pot, e_kin in zip(E_pot, E_kin)]

E_pot_nonlinear = E_pot_nonlin_fun(x_ee_nonlinear[:, 0])
E_kin_nonlinear = E_kin_fun(x_ee_nonlinear[:, 1])
E_total_nonlinear = [e_pot + e_kin for e_pot, e_kin in zip(E_pot_nonlinear, E_kin_nonlinear)]

# Plot results
plot_energy(t, E_pot, E_kin, E_total, E_pot_nonlinear, E_kin_nonlinear, E_total_nonlinear, "Energieerhaltung mit dem expliziten Euler-Verfahren")

### Visualisieren der Ergebnisse

In [ ]:
make_animation(t, x_ee_lin, x_ee_nonlinear, L, delta_t, "Explizites Euler-Verfahren")

## Implizites Euler-Verfahren

Einsetzen der DGL in die Formeln führt auf das folgende nichtlineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - v_{n+1} \cdot \Delta t &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}\sin(\varphi_{n+1}) \Delta t &&= 0
\end{aligned}
$$

Linearisieren von $\sin(\varphi)$ führt auf das lineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - v_{n+1} \cdot \Delta t &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}\varphi_{n+1} \Delta t &&= 0
\end{aligned}
$$

In [ ]:
# Preallocate arrays 
x_ie = x.copy()
x_ie_nonlinear = x.copy()

# Linearized case
A = np.array([[1, -delta_t], [g/L * delta_t, 1]])
inverse = lambda x: 1/(x[0,0]*x[1,1] - x[0,1]*x[1,0]) * np.array([[x[1,1], -x[0,1]], [-x[1, 0], x[0,0]]])   # Returns inverse of a 2x2 matrix

# Nonlinearized case
# Define function f(t_k+1, y_k+1) and system of equations
f_nonlinear_ie = lambda x: np.array([x[1], -g/L * np.sin(x[0])])
sys_of_eq_nonlinear_ie = lambda x_k_1, x_k: x_k_1 - x_k - delta_t * f_nonlinear_ie(x_k_1)                   # = 0
jacobian = lambda x: np.array([[1, -delta_t], [delta_t * g / L * np.cos(x[0]), 1]])                         # Jacobian is determined by hand

# Evaluate for each time step
for n in range(N):
    # Linearized case
    x_ie[n+1] = inverse(A) @ x_ie[n]

    # Nonlinearized Case using newton iteration
    x_guess = x_ie_nonlinear[n]                                 # Start from initial guess (in this case the last timesteps' result)
    for iter in range(5):
        R = sys_of_eq_nonlinear_ie(x_guess, x_ie_nonlinear[n])  # Residual (error from target value of 0)
        J = jacobian(x_guess)                                   # "Direction" of next iteration step
        h = inverse(-J) @ R
        x_guess = x_guess + h

        if norm(h) < 1e-10:
            break

    x_ie_nonlinear[n+1] = x_guess

# Plot results
plot_motion(t, x_ana, x_ie, x_ie_nonlinear, "Implizites Euler-Verfahren")

### Energieerhaltung mit dem impliziten Euler-Verfahren

In [ ]:
# Conservation of Energy
E_pot = E_pot_lin_fun(x_ie[:, 0])
E_kin = E_kin_fun(x_ie[:, 1])
E_total = [e_pot + e_kin for e_pot, e_kin in zip(E_pot, E_kin)]

E_pot_nonlinear = E_pot_nonlin_fun(x_ie_nonlinear[:, 0])
E_kin_nonlinear = E_kin_fun(x_ie_nonlinear[:, 1])
E_total_nonlinear = [e_pot + e_kin for e_pot, e_kin in zip(E_pot_nonlinear, E_kin_nonlinear)]

# Plot results
plot_energy(t, E_pot, E_kin, E_total, E_pot_nonlinear, E_kin_nonlinear, E_total_nonlinear, "Energieerhaltung mit dem impliziten Euler-Verfahren")

### Visualisieren der Ergebnisse

In [ ]:
make_animation(t, x_ie, x_ie_nonlinear, L, delta_t, "Implizites Euler-Verfahren")

## Symplektisches Euler-Verfahren

Einsetzen der DGL in die Formeln führt auf das folgende nichtlineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - v_{n} \cdot \Delta t &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}sin(\varphi_{n+1}) \Delta t &&= 0
\end{aligned}
$$

Linearisieren von $\sin(\varphi)$ führt auf das lineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - v_{n} \cdot \Delta t &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}\varphi_{n+1} \Delta t &&= 0
\end{aligned}
$$

In [ ]:
# Preallocate arrays 
x_se = x.copy()
x_se_nonlinear = x.copy()

# Define functions f(t_k+1, y_k+1) and system of equations
# Linearized case
f_1_se = lambda x: x[1]
f_2_se = lambda x: -g/L * x[0]

# Nonlinearized case
f_1_nonlinear_se = lambda x_k: x_k[1]
f_2_nonlinear_se = lambda x_k_1: -g/L * np.sin(x_k_1[0])

# Evaluate for each time step
for n in range(N):
    # Linearized case
    x_se[n+1, 0] = x_se[n, 0] + f_1_se(x_se[n]) * delta_t
    x_se[n+1, 1] = x_se[n, 1] + f_2_se(x_se[n+1]) * delta_t

    # Nonlinearized Case
    x_se_nonlinear[n+1, 0] = x_se_nonlinear[n, 0] + f_1_nonlinear_se(x_se_nonlinear[n]) * delta_t
    x_se_nonlinear[n+1, 1] = x_se_nonlinear[n, 1] + f_2_nonlinear_se(x_se_nonlinear[n+1]) * delta_t

# Plot results
plot_motion(t, x_ana, x_se, x_se_nonlinear, "Symplektisches Euler-Verfahren")

### Energieerhaltung mit dem symplektischen Euler-Verfahren

In [ ]:
# Conservation of Energy
E_pot = E_pot_lin_fun(x_se[:, 0])
E_kin = E_kin_fun(x_se[:, 1])
E_total = [e_pot + e_kin for e_pot, e_kin in zip(E_pot, E_kin)]

E_pot_nonlinear = E_pot_nonlin_fun(x_se_nonlinear[:, 0])
E_kin_nonlinear = E_kin_fun(x_se_nonlinear[:, 1])
E_total_nonlinear = [e_pot + e_kin for e_pot, e_kin in zip(E_pot_nonlinear, E_kin_nonlinear)]

# Plot results
plot_energy(t, E_pot, E_kin, E_total, E_pot_nonlinear, E_kin_nonlinear, E_total_nonlinear, "Energieerhaltung mit dem symplektischen Euler-Verfahren")

### Visualisieren der Ergebnisse

In [ ]:
make_animation(t, x_se, x_se_nonlinear, L, delta_t, "Symplektisches Euler-Verfahren")

## Trapezregel

Einsetzen der DGL in die Formeln führt auf das folgende nichtlineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - (v_{n} + v_{n+1}) \cdot \frac{\Delta t}{2} &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}(\sin(\varphi_{n}) + \sin(\varphi_{n+1})) \frac{\Delta t}{2} &&= 0
\end{aligned}
$$

Linearisieren von $\sin(\varphi)$ führt auf das lineare Gleichungssystem
$$
\begin{aligned}
&\varphi_{n+1} - \varphi_n - (v_{n} + v_{n+1}) \cdot \frac{\Delta t}{2} &&= 0\\
&v_{n+1} - v_n + \frac{g}{L}(\varphi_n + \varphi_{n+1}) \cdot \frac{\Delta t}{2} && = 0
\end{aligned}
$$

In diesem Abschnitt werden zum Lösen der Gleichungssysteme in Python integrierte Scipy-Funktionen genutzt. Für lineare Gleichungssysteme steht hierfür die Funktion "solve" zur Verfügung. In "solve" wird automatisch zwischen einer LR-Zerlegung oder einer Cholesky-Zerlegung entschieden. 
Vgl. hierzu:
* https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve.html#scipy.linalg.solve
* https://www.netlib.org/lapack/explore-html/d3/d6a/dgetrf_8f_source.html
* https://www.netlib.org/lapack/explore-html/d8/ddb/group__hesv_ga7a2816c471935fe30194282b2b7d06bf.html#ga7a2816c471935fe30194282b2b7d06bf
* https://www.netlib.org/lapack/explore-html/d2/d09/group__potrf.html
* https://www.netlib.org/lapack/explore-html/d8/ddb/group__hesv_gaf54eeccb646de50e6ac0b2eff58db9d5.html#gaf54eeccb646de50e6ac0b2eff58db9d5

Eine in Python geläufige Alternative zum Newton-Verfahren ist die Funktion fsolve der library scipy. Diese basiert auf Powells Hybrid-Methode, einer Variation des Newton-Verfahrens, wobei die Jacobi-Matrix standardmäßig durch Vorwärtsdifferenzen numerisch approximiert wird, jedoch optional auch direkt übergeben werden kann. \
Vgl. auch:
* https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fsolve.html
* https://www.math.utah.edu/software/minpack/minpack/hybrd.html
* https://www.math.utah.edu/software/minpack/minpack/hybrj.html

Die Funktionen "solve" und "fsolve" sind sog. "Wrapper" um etablierte Fortran-Funktionen und daher sehr leistungsstark.

In [ ]:
# Preallocate arrays 
x_tr = x.copy()
x_tr_nonlinear = x.copy()

# Linearized case
A = np.array([[0, 1], [-g/L, 0]])

# Nonlinearized case
# Define functions f(t_k+1, y_k+1) and system of equations
f_nonlinear_tr = lambda x: np.array([x[1], -g/L * np.sin(x[0])])
sys_of_eq_nonlinear_tr = lambda x_k_1, x_k: x_k_1 - x_k - delta_t/2 * (f_nonlinear_tr(x_k_1) + f_nonlinear_tr(x_k))     # = 0

# Evaluate for each time step
for n in range(N):
    # Linearized case
    x_tr[n+1] = solve(np.eye(2) - A  * delta_t/2, (np.eye(2) + A  * delta_t/2) @ x_tr[n])

    # Nonlinearized Case
    x_tr_nonlinear[n+1] = fsolve(sys_of_eq_nonlinear_tr, x_tr_nonlinear[n], args=(x_tr_nonlinear[n]))

# Plot results
plot_motion(t, x_ana, x_tr, x_tr_nonlinear, "Trapezregel")

### Energieerhaltung mit der Trapezregel

In [ ]:
# Conservation of Energy
E_pot = E_pot_lin_fun(x_tr[:, 0])
E_kin = E_kin_fun(x_tr[:, 1])
E_total = [e_pot + e_kin for e_pot, e_kin in zip(E_pot, E_kin)]

E_pot_nonlinear = E_pot_nonlin_fun(x_tr_nonlinear[:, 0])
E_kin_nonlinear = E_kin_fun(x_tr_nonlinear[:, 1])
E_total_nonlinear = [e_pot + e_kin for e_pot, e_kin in zip(E_pot_nonlinear, E_kin_nonlinear)]

# Plot results
plot_energy(t, E_pot, E_kin, E_total, E_pot_nonlinear, E_kin_nonlinear, E_total_nonlinear, "Energieerhaltung mit der Trapezregel")

### Visualisieren der Ergebnisse

In [ ]:
make_animation(t, x_tr, x_tr_nonlinear, L, delta_t, "Trapezregel")

## Konvergenzanalyse
Bei einer Konvergenzanalyse wird der Fehler des Verfahrens in Abhängigkeit der Zeitschrittweite untersucht. Der Fehler ist hier die Differenz zur exakten analytischen Lösung.  

In [ ]:
timesteps = [0.0001, 0.0002, 0.0004, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]
sim_time = 2.0                                  # simualtion time in s

x_start = [phi_0, v_0]                            # initial conditions

# calculate analytical solution
omega = np.sqrt(g/L)
phi_analyt = np.cos(omega*sim_time) * x_start[0]\
             + np.sin(omega*sim_time) * x_start[1]/omega

# initialize arrays to store results
phi_se_end = np.zeros(len(timesteps))
phi_tr_end = np.zeros(len(timesteps))
error_se = np.zeros(len(timesteps))
error_tr = np.zeros(len(timesteps))

# Iterate over each step size
for idx, delta_t in enumerate(timesteps):
    N = int(sim_time / delta_t)                 # Number of time steps                      
    
    # reset the arrays
    x_tr = np.zeros((N+1, 2))
    x_tr[0] = x_start
    x_se = np.zeros((N+1, 2))
    x_se[0,:] = x_start
    
    for n in range(N):

        # Calculate numeric solutions
        x_tr[n+1] = solve(np.eye(2) - A  * delta_t/2, (np.eye(2) + A  * delta_t/2) @ x_tr[n])
        x_se[n+1, 0] = x_se[n, 0] + f_1_se(x_se[n]) * delta_t
        x_se[n+1, 1] = x_se[n, 1] + f_2_se(x_se[n+1]) * delta_t
    
    # Calculate difference between numeric and analytical solution
    phi_se_end[idx] = x_se[-1, 0]
    phi_tr_end[idx] = x_tr[-1, 0]
    error_se[idx] = abs(phi_se_end[idx] - phi_analyt)
    error_tr[idx] = abs(phi_tr_end[idx] - phi_analyt)

plot_convergence(timesteps, error_se, error_tr)    

Die Konvergenzordnung kann als die Steigung der Verbindungslinien im obigen Diagramm abgeschätzt werden. Alternativ kann sie basierend auf drei numerischen Lösungen $\Phi$ mit unterschiedlicher Schrittweite $h$ wie folgt berechnet werden:
$$
p = \log_2 \frac{\Phi _ {2h} - \Phi _{4h}}{\Phi _{h} - \Phi _{2h}}
$$

In [ ]:
p = lambda phi_h, phi_2h, phi_4h: np.log2((phi_2h-phi_4h)/(phi_h-phi_2h))
p_se = p(phi_se_end[0], phi_se_end[1], phi_se_end[2])
p_tr = p(phi_tr_end[0], phi_tr_end[1], phi_tr_end[2])

print(f"Das Symplektische Euler-Verfahren hat die geschätzte Konvergenzordnung p = {p_se}.")
print(f"Die Trapezregel hat die geschätzte Konvergenzordnung p = {p_tr}.")